# Boundary relocation + first-token replacement
Run these cells **inside this notebook**, using your working GPU Python kernel.

Defaults: **128 documents, Qwen2.5 then Qwen3, float32**. Token files are included.
The main run saves each document and runs in the background. Keep the GPU instance on.
These are exploratory extensions on the existing corpus. See `PROTOCOL.txt` for the measurements.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from IPython.display import display, FileLink

candidates = [Path.cwd(), Path.cwd()/"boundary_interventions"]
candidates += list(Path.cwd().glob("*/boundary_interventions"))
ROOT = next((p.resolve() for p in candidates if (p/"run_experiments.py").is_file() and (p/"bundle-integrity.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the extracted boundary_interventions folder.")
os.chdir(ROOT)
(ROOT/"results").mkdir(exist_ok=True)

def run(*arguments, log_name="notebook.log"):
    command = [sys.executable, "-u", *map(str, arguments)]
    print("Running:", " ".join(command), flush=True)
    with (ROOT/"results"/log_name).open("a") as log:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line); log.flush()
            status = process.wait()
        except BaseException:
            process.terminate(); process.wait()
            raise
        finally:
            process.stdout.close()
    if status:
        raise RuntimeError(f"Stopped. The actual error is above; log: results/{log_name}")

print("Project:", ROOT)
print("Python:", sys.executable)
run("setup_environment.py", log_name="setup.log")
run("run_experiments.py", "verify")
run("-m", "unittest", "discover", "-s", "tests", "-v", log_name="tests.log")


## Pilot
This runs both experiments on two validation documents per model, including native attention checks. The first run downloads the models. Models run in subprocesses, leaving no GPU allocations in the notebook kernel. Check the reported timing; do not assume overnight completion.

To change the model list or document count, edit `config.json` **before** running this cell.


In [ ]:
print((ROOT/"config.json").read_text())
run("run_experiments.py", "run", "--stage", "pilot", log_name="pilot.log")
print("Pilot complete. The next cell starts the main run.")


## Start or resume the main run
This cell returns promptly. Collection continues independently of the browser connection.
Do not shut down the GPU instance. Only one collection process should use the GPU.


In [ ]:
run("launch.py")


## Check progress / download results
Rerun this cell for an update. It does not start another experiment.
When complete, download the ZIP below. A numerical-control flag may require a precision follow-up.


In [ ]:
status = ROOT/"results"/"job-status.json"
if status.exists():
    print(status.read_text())
log = ROOT/"results"/"main.log"
if log.exists():
    print("\n".join(log.read_text(errors="replace").splitlines()[-25:]))
archive = ROOT/"results"/"main-intervention-results.zip"
if archive.exists():
    display(FileLink("results/main-intervention-results.zip"))
    for model in json.loads((ROOT/"config.json").read_text())["models"]:
        summary = ROOT/"results"/"main"/model/"summary.json"
        if summary.exists():
            print(model, summary.read_text())
else:
    print("The results ZIP will appear after the main run completes.")
